In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


## Installing Reddis Server

In [2]:
!apt-get update
!apt-get install -y redis-server

Get:1 https://cli.github.com/packages stable InRelease [3,917 B]
Get:2 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ InRelease [3,632 B]
Get:3 https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2204/x86_64  InRelease [1,581 B]
Get:4 http://security.ubuntu.com/ubuntu jammy-security InRelease [129 kB]
Get:5 https://cli.github.com/packages stable/main amd64 Packages [357 B]
Get:6 https://r2u.stat.illinois.edu/ubuntu jammy InRelease [6,555 B]
Get:7 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ Packages [87.4 kB]
Get:8 https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2204/x86_64  Packages [2,452 kB]
Hit:9 http://archive.ubuntu.com/ubuntu jammy InRelease
Get:10 http://archive.ubuntu.com/ubuntu jammy-updates InRelease [128 kB]
Get:11 https://r2u.stat.illinois.edu/ubuntu jammy/main all Packages [9,893 kB]
Get:12 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu jammy InRelease [18.1 kB]
Get:13 http://security.ubuntu.com/ubuntu jammy-secu

##  Starting the reddis server

In [3]:
!redis-server --daemonize yes

## Chechking reddis is running

In [4]:
!redis-cli ping

PONG


## Downloading all the dependencies

In [ ]:
!pip install -U langchain
!pip install -U langchain-community
!pip install -U langchain-classic
!pip install -U langchain-text-splitters
!pip install -U langchain-huggingface
!pip install -U langchain-chroma
!pip install -U chromadb
!pip install -U sentence-transformers
!pip install -U langchain-nvidia-ai-endpoints
!pip install -U rank_bm25
!pip install -U redis
!pip install flask -q


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 112.5/112.5 kB 5.1 MB/s eta 0:00:00
  Attempting uninstall: langchain
    Found existing installation: langchain 1.2.12
    Uninstalling langchain-1.2.12:
      Successfully uninstalled langchain-1.2.12
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 56.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 78.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 64.7/64.7 kB 7.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 51.0/51.0 kB 6.3 MB/s eta 0:00:00
  Attempting uninstall: requests
    Found existing installation: requests 2.32.4
    Uninstalling requests-2.32.4:
      Successfully uninstalled requests-2.32.4
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.32.5 which is incompatible.
     ━━

## Importing all the modules


In [6]:
from langchain_classic.storage import LocalFileStore, create_kv_docstore
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_chroma import Chroma
from langchain_classic.retrievers import ParentDocumentRetriever, BM25Retriever
from langchain_text_splitters import  RecursiveCharacterTextSplitter
from langchain_core.messages import HumanMessage
from langchain_classic.schema import Document
from sentence_transformers import CrossEncoder
from concurrent.futures import ThreadPoolExecutor
from langchain_nvidia_ai_endpoints import ChatNVIDIA
import os, re, base64
import getpass
import textwrap
import json
import logging
import hashlib
import sqlite3
import time
from typing import Optional
import redis

## Initializing the Logger

In [7]:
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s | %(levelname)s | %(message)s",
    datefmt="%H:%M:%S"
)

logger = logging.getLogger(__name__)

## Checking Cuda(Nvidia GPU) is available..


In [8]:
import torch
print(torch.cuda.is_available())

True


In [9]:
import torch
print("CUDA Available:", torch.cuda.is_available())
print("GPU Name:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "No GPU")

CUDA Available: True
GPU Name: Tesla T4


## Defining the embedding Model

In [10]:
local_embeddings = HuggingFaceEmbeddings(
    model_name="BAAI/bge-m3",
    model_kwargs={'device': 'cuda'}, # Use 'cuda' if you have an NVIDIA GPU
    encode_kwargs={'normalize_embeddings': True}
)


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/123 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/54.0 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/444 [00:00<?, ?B/s]

sentencepiece.bpe.model:   0%|          | 0.00/5.07M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/17.1M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/964 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/191 [00:00<?, ?B/s]

## Defining the Cache Class

In [11]:
class QueryCache:
    """Three-tier cache: Query (exact+semantic) + HyDE + LLM Response."""

    def __init__(self, embedding_model, redis_host="localhost", redis_port=6379,
                 redis_db=0, redis_password=None, cache_dir="./cache",
                 semantic_threshold=0.92, ttl_seconds=86400 * 7, key_prefix="rag_cache:"):

        self.embedding_model = embedding_model
        self.cache_dir = cache_dir
        self.semantic_threshold = semantic_threshold
        self.ttl_seconds = ttl_seconds
        self.key_prefix = key_prefix

        self.redis = redis.Redis(
            host=redis_host, port=redis_port, db=redis_db,
            password=redis_password, decode_responses=True
        )

        try:
            self.redis.ping()
            print(f"✅ Connected to Redis at {redis_host}:{redis_port}")
        except redis.ConnectionError:
            raise ConnectionError(
                f"❌ Cannot connect to Redis at {redis_host}:{redis_port}. "
                f"Run: sudo systemctl start redis"
            )

        os.makedirs(cache_dir, exist_ok=True)
        self.semantic_store = Chroma(
            collection_name="query_cache_semantic",
            embedding_function=embedding_model,
            persist_directory=os.path.join(cache_dir, "semantic_db"),
        )

        count = len(self.redis.keys(f"{self.key_prefix}*"))
        print(f"✅ QueryCache initialized — {count} cached entries in Redis")

    @staticmethod
    def _hash_query(query):
        normalized = query.strip().lower()
        return hashlib.sha256(normalized.encode("utf-8")).hexdigest()

    def _redis_key(self, prefix, hash_val):
        return f"{self.key_prefix}{prefix}:{hash_val}"

    # ─────────────────────────────────────────────
    # LAYER 1: Query Cache (Exact + Semantic)
    # ─────────────────────────────────────────────

    def get_query(self, query):
        """Layer 1: Check exact match, then semantic match."""
        query_hash = self._hash_query(query)
        key = self._redis_key("query", query_hash)

        # Exact match
        cached = self.redis.get(key)
        if cached:
            result = json.loads(cached)
            result["_cache_type"] = "exact"
            print(f" Layer 1 HIT (exact match)")
            return result

        # Semantic match
        try:
            results = self.semantic_store.similarity_search_with_relevance_scores(query, k=1)
            if results:
                doc, score = results[0]
                if score >= self.semantic_threshold:
                    similar_hash = doc.metadata.get("query_hash")
                    cached = self.redis.get(self._redis_key("query", similar_hash))
                    if cached:
                        result = json.loads(cached)
                        result["_cache_type"] = "semantic"
                        result["_similar_query"] = doc.page_content
                        result["_similarity_score"] = round(score, 4)
                        print(f"⚡ Layer 1 HIT (semantic, score={score:.4f})")
                        print(f"   Matched: \"{doc.page_content}\"")
                        return result
        except Exception:
            pass

        print(f"❌ Layer 1 MISS")
        return None

    def put_query(self, query, response):
        """Layer 1: Store final response."""
        query_hash = self._hash_query(query)
        clean = {k: v for k, v in response.items() if not k.startswith("_")}

        self.redis.setex(
            self._redis_key("query", query_hash),
            self.ttl_seconds,
            json.dumps(clean)
        )

        self.semantic_store.add_texts(
            texts=[query.strip()],
            metadatas=[{"query_hash": query_hash}],
            ids=[query_hash],
        )
        print(f"💾 Layer 1 cached: \"{query[:60]}\"")

    # ─────────────────────────────────────────────
    # LAYER 2: HyDE Vector Cache
    # ─────────────────────────────────────────────

    def get_hyde(self, query):
        """Layer 2: Check if HyDE vector for this query is cached."""
        query_hash = self._hash_query(query)
        key = self._redis_key("hyde", query_hash)

        cached = self.redis.get(key)
        if cached:
            print(f"⚡ Layer 2 HIT (HyDE vector cached)")
            return json.loads(cached)

        print(f"❌ Layer 2 MISS (HyDE)")
        return None

    def put_hyde(self, query, hyde_vector):
        """Layer 2: Store HyDE vector."""
        query_hash = self._hash_query(query)
        key = self._redis_key("hyde", query_hash)

        self.redis.setex(key, self.ttl_seconds, json.dumps(hyde_vector))
        print(f"💾 Layer 2 cached: HyDE vector")

    # ─────────────────────────────────────────────
    # LAYER 3: LLM Response Cache
    # ─────────────────────────────────────────────

    def get_llm_response(self, query, context_docs):
        """Layer 3: Check if LLM already answered this query with this context."""
        # Hash query + context together
        context_str = json.dumps(context_docs, sort_keys=True, default=str)
        combined = f"{query.strip().lower()}||{context_str}"
        combined_hash = hashlib.sha256(combined.encode("utf-8")).hexdigest()
        key = self._redis_key("llm", combined_hash)

        cached = self.redis.get(key)
        if cached:
            print(f"⚡ Layer 3 HIT (LLM response cached)")
            return json.loads(cached)

        print(f"❌ Layer 3 MISS (LLM)")
        return None

    def put_llm_response(self, query, context_docs, response):
        """Layer 3: Store LLM response keyed by query + context."""
        context_str = json.dumps(context_docs, sort_keys=True, default=str)
        combined = f"{query.strip().lower()}||{context_str}"
        combined_hash = hashlib.sha256(combined.encode("utf-8")).hexdigest()
        key = self._redis_key("llm", combined_hash)

        clean = {k: v for k, v in response.items() if not k.startswith("_")}
        self.redis.setex(key, self.ttl_seconds, json.dumps(clean))
        print(f"💾 Layer 3 cached: LLM response")

    # ─────────────────────────────────────────────
    # UTILITIES
    # ─────────────────────────────────────────────

    def clear(self):
        """Flush all cache layers."""
        keys = self.redis.keys(f"{self.key_prefix}*")
        if keys:
            self.redis.delete(*keys)
        self.semantic_store.delete_collection()
        self.semantic_store = Chroma(
            collection_name="query_cache_semantic",
            embedding_function=self.embedding_model,
            persist_directory=os.path.join(self.cache_dir, "semantic_db"),
        )
        print("🗑️ All cache layers cleared")

    def stats(self):
        """Return per-layer cache statistics."""
        all_keys = self.redis.keys(f"{self.key_prefix}*")
        query_keys = [k for k in all_keys if ":query:" in k]
        hyde_keys = [k for k in all_keys if ":hyde:" in k]
        llm_keys = [k for k in all_keys if ":llm:" in k]
        return {
            "layer_1_query_cache": len(query_keys),
            "layer_2_hyde_cache": len(hyde_keys),
            "layer_3_llm_cache": len(llm_keys),
            "total": len(all_keys),
        }

## Intializing the Cache Object

In [12]:
## Initialize Cache

cache = QueryCache(
    embedding_model=local_embeddings,
    redis_host="localhost",
    redis_port=6379,
    cache_dir="./cache",
    semantic_threshold=0.92,
    ttl_seconds=86400 * 7
)

✅ Connected to Redis at localhost:6379
✅ QueryCache initialized — 0 cached entries in Redis


## defining current dir

In [13]:
current_dir = os.getcwd()

In [14]:
current_dir = f"{current_dir}/drive/MyDrive/ACM_Hackathon_Project"
current_dir

'/content/drive/MyDrive/ACM_Hackathon_Project'

## Loading the VectorDb

In [15]:
persist_dir = f"{current_dir}/chroma_db_local/chroma_db_local"
parent_store_dir = f"{current_dir}/parent_store_local/parent_store_local"

# Load the Persistent Parent and Child Storage
fs = LocalFileStore(parent_store_dir)
store = create_kv_docstore(fs)

# Setup Chroma as before
vectorstore = Chroma(
    collection_name="med_rag_local",
    embedding_function=local_embeddings,
    persist_directory=persist_dir
)


child_splitter = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=100)
# 4. Initialize Retriever with the PERSISTENT store
retriever = ParentDocumentRetriever(
    vectorstore=vectorstore,
    docstore=store,
    child_splitter=child_splitter
)

## Initializing the llms

In [16]:

if "NVIDIA_API_KEY" in os.environ:
    del os.environ["NVIDIA_API_KEY"]
if "NVIDIA_API_KEY" not in os.environ:
    os.environ["NVIDIA_API_KEY"] = getpass.getpass("Enter your NVIDIA NIM API Key: ")

# 2. Initialize the Best Free LLM (Llama 3.3 70B Instruct)
print("🚀 Initializing Llama-3.3-70B via NVIDIA API...")
llm = ChatNVIDIA(
    model="meta/llama-3.3-70b-instruct",
    temperature=0.2, # Keep it low for factual textbook answers
    max_completion_tokens=1024
)


print("🚀 Initializing Llama-3.2-90B-Vision via NVIDIA API...")
vision_llm = ChatNVIDIA(
    model="meta/llama-3.1-70b-instruct",
    temperature=0.2,
    max_completion_tokens=1024
)

Enter your NVIDIA NIM API Key: ··········
🚀 Initializing Llama-3.3-70B via NVIDIA API...
🚀 Initializing Llama-3.2-90B-Vision via NVIDIA API...


## Implementing HyDE (Hypothethical Document Embedding)

In [17]:
def generate_hyde_vector(query: str, llm_client, embedding_model) -> list[float]:
    """
    Generates a hypothetical document and returns its vector embedding.

    Args:
        query: The user's search query.
        llm_client: Your initialized LLM (e.g., your Llama generator function).
        embedding_model: Your initialized BGE-M3 embedder.
    """

    # 1. The Prompt: Instruct the LLM to write a textbook-style answer
    prompt = f"""Given the following question, write a hypothetical, detailed textbook passage that directly answers it.
    Write in an academic tone, use standard formatting, and include relevant domain keywords that might appear in a textbook.

    Question: {query}

    Hypothetical Textbook Passage:"""

    # 2. Generate the hypothetical document
    # (Adjust the calling method based on your specific LLM wrapper)

    response = llm_client.invoke([
        HumanMessage(content=prompt)
    ])

    hypothetical_doc = response.content
    # print(f"--- Hypothetical Document Generated ---\n{hypothetical_doc}\n---------------------------------------")

    # 3. Embed the generated document instead of the raw query
    # (Adjust .encode() based on whether you are using HuggingFace, SentenceTransformers, etc.)
    hyde_vector = embedding_model.embed_query(hypothetical_doc)

    return hyde_vector

# --- Usage Example ---
# user_query = "What are the prigenerate_hyde_vectormary functions of the sympathetic nervous system?"
# query_vector = (user_query, my_llama_model, my_bge_m3_model)
#
# # Now you pass `query_vector` into your Vector DB for the similarity search!

## Implementing the normal Query embedding

In [18]:
def get_query_vector(query, embedding_model, llm=None, use_hyde=True):
    """
    Generate query vector using either HyDE or direct embedding.

    Args:
        query: The user's question
        embedding_model: BGE-M3 embedding model
        llm: LLM for HyDE generation (only needed if use_hyde=True)
        use_hyde: True = HyDE (slower, better retrieval) | False = direct embed (faster)

    Returns:
        query_vector (list[float])
    """
    if use_hyde:
        if llm is None:
            raise ValueError("LLM is required when use_hyde=True")
        print("🔍 Using HyDE (generating hypothetical passage)...")
        query_vector = generate_hyde_vector(query, llm, embedding_model)
    else:
        print("⚡ Using direct query embedding (fast mode)...")
        query_vector = embedding_model.embed_query(query)

    return query_vector

## Initializing the BM25_Retriever

In [19]:


# Extract all child chunks directly from Chroma
# Calling .get() provides a dictionary output containing the 'documents' and 'metadatas' arrays.
chroma_data = vectorstore.get()

# Reconstruct the LangChain Document objects so BM25 can read them
child_chunks = []
for i in range(len(chroma_data['documents'])):
    doc = Document(
        page_content=chroma_data['documents'][i],
        metadata=chroma_data['metadatas'][i] if chroma_data['metadatas'] else {}
    )
    child_chunks.append(doc)

print(f"Loaded {len(child_chunks)} child chunks directly from ChromaDB.")

# 3. Instantiate the BM25Retriever using the extracted chunks
my_bm25_retriever = BM25Retriever.from_documents(child_chunks)

Loaded 7775 child chunks directly from ChromaDB.


## Define and Implementing the BM25 Search and Vector Search
    

In [20]:

def retrieve_and_fuse(query, hyde_vector, bm25_retriever, vector_store, top_k=5):
    """Hybrid search with PARALLEL BM25 + Vector retrieval, fused using RRF."""

    bm25_retriever.k = top_k

    # Run both searches simultaneously on 2 threads
    with ThreadPoolExecutor(max_workers=2) as executor:
        bm25_future = executor.submit(bm25_retriever.invoke, query)
        vector_future = executor.submit(vector_store.similarity_search_by_vector, hyde_vector, k=top_k)

        bm25_docs = bm25_future.result()
        vector_docs = vector_future.result()

    print(f"Retrieved {len(bm25_docs)} BM25 + {len(vector_docs)} Vector docs (parallel)")

    # Reciprocal Rank Fusion
    fused_scores = {}
    rrf_k = 60

    def score_docs(docs):
        for rank, doc in enumerate(docs):
            doc_id = doc.page_content
            if doc_id not in fused_scores:
                fused_scores[doc_id] = {"doc": doc, "score": 0.0}
            fused_scores[doc_id]["score"] += 1.0 / (rank + rrf_k)

    score_docs(bm25_docs)
    score_docs(vector_docs)

    reranked = sorted(fused_scores.values(), key=lambda x: x["score"], reverse=True)
    final_docs = [item["doc"] for item in reranked]

    print(f"Fused into {len(final_docs)} unique child chunks.")
    return final_docs

## Implementing the cross-encoder re-ranker from BGE Ecosystem bge-reranker-large


In [ ]:

# 1. Initialize the Cross-Encoder model
# (Note: This will download the model the first time you run it. Switch to 'BAAI/bge-reranker-base' if 'large' is too heavy for your RAM limits)
bge_reranker = CrossEncoder("BAAI/bge-reranker-large", device="cuda")

def rerank_chunks(query: str, fused_docs: list, reranker_model, top_n: int = 3, min_score=0.01) -> list:
    """Rerank with score threshold to filter low-relevance chunks."""
    if not fused_docs:
        return []
    pairs = [[query, doc.page_content] for doc in fused_docs]
    scores = reranker_model.predict(pairs)
    scored_docs = [(s, d) for s, d in zip(scores, fused_docs) if s > min_score]
    scored_docs.sort(key=lambda x: x[0], reverse=True)
    kept = [doc for _, doc in scored_docs[:top_n]]
    filtered_out = len(fused_docs) - len(scored_docs)
    if filtered_out > 0:
        print(f"  🔍 Filtered out {filtered_out} low-relevance chunks")
    return kept


config.json:   0%|          | 0.00/801 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/2.24G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/393 [00:00<?, ?it/s]

XLMRobertaForSequenceClassification LOAD REPORT from: BAAI/bge-reranker-large
Key                             | Status     |  | 
--------------------------------+------------+--+-
roberta.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


tokenizer_config.json:   0%|          | 0.00/443 [00:00<?, ?B/s]

sentencepiece.bpe.model:   0%|          | 0.00/5.07M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/17.1M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/279 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

## Fetch and Deduplicate the Parent Chunk


In [22]:
parent_store = LocalFileStore(f"{current_dir}/parent_store_local/parent_store_local")


def fetch_and_deduplicate_parents(top_child_chunks: list, store) -> list:
    """Maps child chunks to parents and drops duplicates. Now includes page metadata."""
    unique_parent_ids = set()
    final_parent_docs = []

    for child in top_child_chunks:
        parent_id = child.metadata.get('doc_id')

        if parent_id and parent_id not in unique_parent_ids:
            unique_parent_ids.add(parent_id)

            parent_bytes = store.mget([parent_id])[0]

            if parent_bytes:
                parent_text = parent_bytes.decode('utf-8')
                # Carry forward the metadata from child → parent result
                final_parent_docs.append({
                    "text": parent_text,
                    "chapter": child.metadata.get("chapter", "Unknown"),
                    "section": child.metadata.get("section", "Unknown"),
                    "section_title": child.metadata.get("section-title", "Unknown"),
                    "page_start": child.metadata.get("page_start", "N/A"),
                    "page_end": child.metadata.get("page_end", "N/A"),
                })

    print(f"Reduced {len(top_child_chunks)} child chunks to {len(final_parent_docs)} unique parent chunks.")
    return final_parent_docs

In [23]:
def encode_image_to_base64(image_path: str) -> str:
    with open(image_path, "rb") as image_file:
        return base64.b64encode(image_file.read()).decode('utf-8')

## Generating the final answer

In [ ]:
def generate_final_answer(query, parent_docs, vision_llm_client):
    """Generate answer with strict faithfulness guardrails."""

    formatted_contexts = []
    for i, doc in enumerate(parent_docs):
        source_tag = f"[Source {i+1}]"
        meta_line = (
            f"Chapter: {doc['chapter']} | Section: {doc['section']} "
            f"| Title: {doc['section_title']} | Pages: {doc['page_start']}-{doc['page_end']}"
        )
        formatted_contexts.append(f"--- {source_tag} ({meta_line}) ---\n{doc['text']}\n")

    combined_context = "\n".join(formatted_contexts)
    image_paths = re.findall(r'!\[[^\]]*\]\(([^)]+)\)', combined_context)

    json_schema = textwrap.dedent("""\
    {
      "answer": "Detailed answer with inline citations like [Source 1].",
      "citations": [
        {
          "source_tag": "[Source 1]",
          "chapter": "Chapter name",
          "section": "Section number",
          "section_name": "Title",
          "page_start": 1,
          "page_end": 2
        }
      ]
    }""")

    system_prompt = f"""You are an expert academic assistant.

STRICT RULES:
1. ONLY use information from the provided textbook context below. DO NOT use your own knowledge.
2. Every factual claim MUST have an inline citation like [Source 1].
3. If a claim cannot be directly supported by the context, DO NOT include it.
4. If the context does not contain enough information to answer fully, explicitly state: "The retrieved sections do not contain sufficient information to fully answer this question."
5. Prefer quoting or closely paraphrasing the textbook rather than adding your own interpretation.
6. Structure your response in the JSON format below.

JSON Schema:
{json_schema}

Textbook Context:
{combined_context}

Question: {query}

Answer (use ONLY the textbook context above, cite every claim):"""

    message_content = [
        {"type": "text", "text": system_prompt}
    ]

    for path in image_paths:
        if os.path.exists(path):
            base64_image = encode_image_to_base64(path)
            message_content.append({
                "type": "image_url",
                "image_url": {"url": f"data:image/jpeg;base64,{base64_image}"}
            })

    print(f"Passing context ({len(parent_docs)} sources, {len(image_paths)} images) to Llama Vision...")
    response = vision_llm_client.invoke([HumanMessage(content=message_content)])
    return response.content


## Calling the All the above function to get the answer for the query

In [25]:
def run_multimodal_rag_pipeline(
    query,
    llm,
    vision_llm,
    bge_m3_embedder,
    bm25_index,
    vector_db,
    bge_reranker,
    parent_retriever
):
    logger.info("🚀 Starting Multimodal RAG Pipeline")
    logger.info(f"🔎 Query: {query}")

    try:
        logger.info("1️⃣ Generating HyDE vector...")
        hyde_vec = generate_hyde_vector(query, llm, bge_m3_embedder)
        logger.info("✅ HyDE vector generated successfully")

        logger.info("2️⃣ Retrieving and fusing child chunks (BM25 + Vector)...")
        fused_children = retrieve_and_fuse(query, hyde_vec, bm25_index, vector_db)
        logger.info(f"✅ Retrieved & fused {len(fused_children)} child chunks")

        logger.info("3️⃣ Re-ranking fused chunks with Cross-Encoder...")
        top_children = rerank_chunks(query, fused_children, bge_reranker)
        logger.info(f"✅ Re-ranked top {len(top_children)} chunks")

        logger.info("4️⃣ Fetching and deduplicating Parent Chunks...")
        final_parents = fetch_and_deduplicate_parents(top_children, parent_retriever)
        logger.info(f"✅ Retrieved {len(final_parents)} unique parent chunks")

        logger.info("5️⃣ Generating final multimodal answer...")
        answer = generate_final_answer(query, final_parents, vision_llm)

        logger.info("🎉 Pipeline completed successfully")

        return answer

    except Exception as e:
        logger.exception("❌ Error occurred during RAG pipeline execution")
        raise

## Displaying it in Beutiful Way

In [26]:
def display_rag_answer(result: dict, width: int = 100):
    """Pretty prints RAG answer + citations with page numbers."""

    print("\n" + "=" * width)
    print("📘 FINAL ANSWER".center(width))
    print("=" * width + "\n")

    wrapped_answer = textwrap.fill(result["answer"], width=width)
    print(wrapped_answer)

    print("\n" + "-" * width)
    print("📚 SOURCES".center(width))
    print("-" * width + "\n")

    for idx, citation in enumerate(result["citations"], 1):
        print(f"{idx}. {citation['source_tag']}")
        print(f"   📖 Chapter : {citation['chapter']}")
        print(f"   📑 Section : {citation['section']}")
        print(f"   🏷️ Title   : {citation['section_name']}")
        print(f"   📄 Pages   : {citation.get('page_start', 'N/A')} - {citation.get('page_end', 'N/A')}")
        print()

    print("=" * width + "\n")

In [ ]:
USE_HYDE = True
TOP_K = 7
TOP_N_RERANK = 3


def give_final_answer(query):
    # Layer 1: Full response cache
    cached = cache.get_query(query)
    if cached:
        return cached

    # Layer 2: HyDE cache
    hyde_vec = cache.get_hyde(query)
    if hyde_vec is None:
        hyde_vec = get_query_vector(query, local_embeddings, llm, use_hyde=USE_HYDE)
        cache.put_hyde(query, hyde_vec)

    # Retrieval + Reranking (TOP_K=7, min_score threshold)
    fused_children = retrieve_and_fuse(query, hyde_vec, my_bm25_retriever, vectorstore, top_k=TOP_K)
    top_children = rerank_chunks(query, fused_children, bge_reranker, top_n=TOP_N_RERANK, min_score=0.01)
    final_parents = fetch_and_deduplicate_parents(top_children, parent_store)

    # Layer 3: LLM response cache
    result = cache.get_llm_response(query, final_parents)
    if result is None:
        final_answer = generate_final_answer(query, final_parents, vision_llm)
        result = json.loads(final_answer)
        cache.put_llm_response(query, final_parents, result)

    # Store in Layer 1
    cache.put_query(query, result)
    return result


In [33]:
query = "Which of the research methods discussed in this section would be best suited to study the impact of diet and exercise on the prevalence of a disease such as diabetes? Why?"
result = give_final_answer(query)
display_rag_answer(result)

⚡ Layer 1 HIT (semantic, score=0.9713)
   Matched: "Which of the research methods discussed in this section would be best suited to study the impact of diet and exercise on the prevalence of a disease such as diabetes? Why?"

                                           📘 FINAL ANSWER                                           

Longitudinal research would be best suited to study the impact of diet and exercise on the
prevalence of a disease such as diabetes. This is because longitudinal research involves tracking
the same group of individuals repeatedly over an extended period of time, allowing researchers to
examine how changes in diet and exercise habits affect the development of diabetes over time [Source
2].

----------------------------------------------------------------------------------------------------
                                             📚 SOURCES                                              
-----------------------------------------------------------------------------

## Generating a Submission.csv File

In [41]:
import json
import csv

with open(f"{current_dir}/docs/queries.json", "r", encoding="utf-8") as f:
    queries = json.load(f)

total_queries = len(queries)
print(f"🚀 Starting processing of {total_queries} queries...\n")

with open("submission.csv", "w", newline="", encoding="utf-8") as csvfile:
    fieldnames = ["query_id", "question", "answer", "citations"]
    writer = csv.DictWriter(csvfile, fieldnames=fieldnames)
    writer.writeheader()

    for idx, item in enumerate(queries, start=1):
        query_id = item["query_id"]
        question = item["question"]

        print(f"⏳ Processing Query {idx}/{total_queries} (ID: {query_id})...")
        result = give_final_answer(question)

        answer_text = result["answer"]

        citation_texts = []
        for c in result["citations"]:
            citation_texts.append(
                f"{c['source_tag']} | Chapter: {c['chapter']} | "
                f"Section: {c['section']} | Title: {c['section_name']} | "
                f"Pages: {c.get('page_start', 'N/A')}-{c.get('page_end', 'N/A')}"
            )

        citations_combined = " || ".join(citation_texts)

        writer.writerow({
            "query_id": query_id,
            "question": question,
            "answer": answer_text,
            "citations": citations_combined
        })

        print(f"✅ Query {idx} done\n")

print("🎉 submission.csv created successfully!")

🚀 Starting processing of 50 queries...

⏳ Processing Query 1/50 (ID: 1)...
❌ Layer 1 MISS
❌ Layer 2 MISS (HyDE)
💾 Layer 2 cached: HyDE vector
Retrieved 5 BM25 docs and 5 Vector docs.
Successfully fused into 9 unique child chunks.
Reduced 3 child chunks to 3 unique parent chunks.
❌ Layer 3 MISS (LLM)
Passing context (with 3 citations) and 4 images to Llama Vision...
💾 Layer 3 cached: LLM response
💾 Layer 1 cached: "What is the scientific method in psychology?"
✅ Query 1 done

⏳ Processing Query 2/50 (ID: 2)...
❌ Layer 1 MISS
❌ Layer 2 MISS (HyDE)
💾 Layer 2 cached: HyDE vector
Retrieved 5 BM25 docs and 5 Vector docs.
Successfully fused into 9 unique child chunks.
Reduced 3 child chunks to 2 unique parent chunks.
❌ Layer 3 MISS (LLM)
Passing context (with 2 citations) and 2 images to Llama Vision...
💾 Layer 3 cached: LLM response
💾 Layer 1 cached: "What are the basic parts of a neuron?"
✅ Query 2 done

⏳ Processing Query 3/50 (ID: 3)...
 Layer 1 HIT (exact match)
✅ Query 3 done

⏳ Process

/tmp/ipykernel_512/208166811.py:65: UserWarning: Relevance scores must be between 0 and 1, got [(Document(id='4cd3844e7ed035575ce42e8500ce7b9a8f859b06890361182577de1fdfcf9f82', metadata={'query_hash': '4cd3844e7ed035575ce42e8500ce7b9a8f859b06890361182577de1fdfcf9f82'}, page_content='What is social psychology?'), -0.028896199399585987)]
  results = self.semantic_store.similarity_search_with_relevance_scores(query, k=1)


💾 Layer 2 cached: HyDE vector
Retrieved 5 BM25 docs and 5 Vector docs.
Successfully fused into 7 unique child chunks.
Reduced 3 child chunks to 3 unique parent chunks.
❌ Layer 3 MISS (LLM)
Passing context (with 3 citations) and 2 images to Llama Vision...
💾 Layer 3 cached: LLM response
💾 Layer 1 cached: "Who were Wilhelm Wundt and William James?"
✅ Query 12 done

⏳ Processing Query 13/50 (ID: 13)...
❌ Layer 1 MISS
❌ Layer 2 MISS (HyDE)
💾 Layer 2 cached: HyDE vector
Retrieved 5 BM25 docs and 5 Vector docs.
Successfully fused into 10 unique child chunks.
Reduced 3 child chunks to 3 unique parent chunks.
❌ Layer 3 MISS (LLM)
Passing context (with 3 citations) and 1 images to Llama Vision...
💾 Layer 3 cached: LLM response
💾 Layer 1 cached: "What is functionalism in psychology?"
✅ Query 13 done

⏳ Processing Query 14/50 (ID: 14)...
❌ Layer 1 MISS
❌ Layer 2 MISS (HyDE)
💾 Layer 2 cached: HyDE vector
Retrieved 5 BM25 docs and 5 Vector docs.
Successfully fused into 10 unique child chunks.
Reduc

In [ ]:
# ============================================
# 🌐 HTTP Server — RAG API Endpoint
# ============================================
# Run this cell AFTER all models and pipeline functions are loaded.
# The server will run on port 8000 and accept POST requests.

!pip install flask -q

from flask import Flask, request, jsonify
import threading
import time

app = Flask(__name__)

@app.route("/api/health", methods=["GET"])
def health():
    return jsonify({"status": "ok", "message": "RAG pipeline is running"})


@app.route("/api/query", methods=["POST"])
def handle_query():
    """
    Accepts: {"query": "What is operant conditioning?"}
    Returns: {"answer": "...", "citations": [...], "latency": 5.2}
    """
    data = request.get_json()
    query = data.get("query", "").strip()

    if not query:
        return jsonify({"error": "query is required"}), 400

    print(f"\n🔍 Incoming query: {query}")
    t0 = time.time()

    try:
        result = give_final_answer(query)
        latency = round(time.time() - t0, 2)

        print(f"✅ Answered in {latency}s")

        return jsonify({
            "success": True,
            "answer": result.get("answer", ""),
            "citations": result.get("citations", []),
            "latency": latency,
        })

    except Exception as e:
        print(f"❌ Error: {e}")
        return jsonify({"success": False, "error": str(e)}), 500


# Run in a background thread so the notebook doesn't block
def run_server():
    app.run(host="0.0.0.0", port=8000, debug=False, use_reloader=False)

server_thread = threading.Thread(target=run_server, daemon=True)
server_thread.start()

print("🚀 RAG API server running at http://localhost:8000")
print("   POST /api/query   → {\"query\": \"your question\"}")
print("   GET  /api/health  → health check")
